# Relative Flux Light Curves per DIA Object for PSFFlux and ApFlux

This notebook loads the enriched alert table produced by `03_associateFinkAlerts-RubinVisits.ipynb`
(`src_joined_butler.parquet` preferred, or `src_joined_consdb.parquet` as fallback)
and displays, for the **N top-ranked DIA objects** (sorted by decreasing alert count):

1. **Relative flux** = `psfFlux(t) / median(psfFlux)` per band, with error bars,
   as a function of **relative time** w.r.t. the first alert of the object.
   The subplots are arranged as: `u | g | r | i | z | y | all-bands`.

2. **PSF-flux ratio vs. AP-flux ratio** per band — same subplot layout.

The x-axis bottom gives `Δt [days]` from the first alert; the x-axis top gives calendar
dates (`YYYY-MM-DD`) so the observer can instantly locate the observation epoch.

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Last update:** 2026-04-01
- **Last update:** 2026-04-06  
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend (zoom/pan) when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_05"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"  # input: written by notebook 03
DIR_FIGS = f"figs_{NB_TAG}"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (butler preferred; fallback to consdb) ───────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

# ── Number of top-ranked DIA objects to plot ──────────────────────────────────
N_TOP = 20  # <── change here

# ── Band definitions ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names expected in the parquet file ─────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_AP = "r:apFlux"
COL_APERR = "r:apFluxErr"


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load data

In [ ]:
# Butler preferred; fallback to consDb
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Source label: {src_label}")

In [ ]:
# df = pd.read_parquet(FILE_CONSDB)
# src_label = "consdb"
# print(f"Loaded consDb file: {FILE_CONSDB}")
# print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
# print(f"Source label: {src_label}")

## 3. Schema inspection & column validation

In [ ]:
print("Columns in loaded table:")
print(df.dtypes.to_string())

In [ ]:
# Validate required columns
required_cols = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

# Optional AP flux
has_ap = (COL_AP in df.columns) and (COL_APERR in df.columns)
if not has_ap:
    print("WARNING: apFlux / apFluxErr columns not found — PSF-vs-AP comparison will be skipped.")

print("All required columns present.")
print(f"apFlux available : {has_ap}")

## 4. Rank DIA objects by number of alerts (decreasing)

In [ ]:
# Count alerts per diaObjectId and sort descending
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)

print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
# Select top-N objects
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()

# Ensure MJD is float
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

print(f"Rows kept for top-{N_TOP} objects : {len(df_top):,}")

## 5. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """
    Convert an array of MJD (float) values to ISO date strings 'YYYY-MM-DD'.
    Uses astropy.time.Time for accuracy.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_relative_flux(flux, flux_err):
    """
    Compute  ratio = flux / median(flux)  and propagate errors.

    Parameters
    ----------
    flux     : array-like  — measured fluxes (nJy)
    flux_err : array-like  — 1-sigma flux uncertainties (nJy)

    Returns
    -------
    ratio      : ndarray   — flux / median(flux)
    ratio_err  : ndarray   — propagated 1-sigma uncertainty on ratio
    flux_median: float     — the median flux used for normalisation
    sigma_ratio: float     — RMS scatter of the ratio (robustness indicator)
    """
    f = np.asarray(flux, dtype=float)
    fe = np.asarray(flux_err, dtype=float)

    # Use finite positive fluxes for median computation
    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), np.nan, np.nan

    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), 0.0, np.nan

    ratio = f / f_med
    ratio_err = fe / f_med  # median treated as a constant → sigma_ratio = sigma_flux / median

    # Scatter of the ratio
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))

    return ratio, ratio_err, f_med, sigma_ratio


print("Utility functions defined.")

## 6. Loop: relative-flux light curves (PSF and AP separately, per band)

For each DIA object:
- 7 subplots: one per band `u g r i z y` + one combined panel
- x bottom : relative time Δt [days] from first alert
- x top    : calendar date (YYYY-MM-DD)
- y        : ratio = flux(t) / median_flux

In [ ]:
def plot_relative_flux_object(
    obj_id, df_obj, flux_col, flux_err_col, flux_label, field_col, ax_rows, yscale_min=0.5, yscale_max=2.0
):
    """
    Plot relative flux light curves for one DIA object.

    Parameters
    ----------
    obj_id     : int / str  — diaObjectId
    df_obj     : DataFrame  — rows for this object only
    flux_col   : str        — flux column name (psfFlux or apFlux)
    flux_err_col: str       — flux error column name
    flux_label : str        — label for the title/y-axis
    field_col  : str        - DDF name
    ax_rows    : list of Axes — 7 axes [u, g, r, i, z, y, combined]
    """
    # Global t0 = MJD of the very first alert across all bands
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    field = df_obj[field_col]

    # Per-band loop
    idx_firstband = -1
    for idx, band in enumerate(BANDS):
        ax = ax_rows[idx]
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)

        if len(df_b) == 0:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        # add info on the leftmost band
        if idx_firstband != -1:
            idx_firstband = idx

        dt = df_b[COL_MJD].values - t0_mjd  # relative time [days]
        ratio, ratio_err, f_med, sigma = compute_relative_flux(
            df_b[flux_col].values, df_b[flux_err_col].values
        )

        dt_min = dt.min() - 1.0
        dt_max = dt.max() + 1.0

        color = BAND_COLORS.get(band, "k")
        ax.errorbar(
            dt,
            ratio,
            yerr=ratio_err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=2,
            alpha=0.85,
            label=f"{band}  σ={sigma:.3f}",
        )

        # force a common x-scale
        ax.set_xlim(dt_min, dt_max)
        ax.set_ylim(yscale_min, yscale_max)

        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}  |  σ(ratio)={sigma:.3f}", fontsize=9)
        ax.set_ylabel(f"{flux_label} / median", fontsize=8)

        # if idx == idx_firstband:
        # if idx == 0:
        #    if field is not None:
        #        y_label = f"diaOb:{obj_id}({field})\n{flux_label} / median"
        #    else:
        #        y_label = f"diaOb:{obj_id}\n{flux_label} / median"
        #    ax.set_ylabel(y_label, fontsize=8)
        # else:
        #    ax.set_ylabel(f"{flux_label} / median", fontsize=8)

        # Top x-axis: calendar dates
        # Sample a few tick positions in dt space and label them
        _add_date_axis(ax, dt, t0_mjd)

        # force a common x-scale
        ax.set_xlim(dt_min, dt_max)
        ax.set_ylim(yscale_min, yscale_max)

    # ── Combined panel (last subplot) ──────────────────────────────────────────
    ax_all = ax_rows[-1]
    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        ratio, ratio_err, _, sigma = compute_relative_flux(df_b[flux_col].values, df_b[flux_err_col].values)
        color = BAND_COLORS.get(band, "k")
        ax_all.errorbar(
            dt,
            ratio,
            yerr=ratio_err,
            fmt="o",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.7,
            label=f"{band} σ={sigma:.3f}",
        )

    # force a common x-scale
    ax_all.set_xlim(dt_min, dt_max)
    ax_all.set_ylim(yscale_min, yscale_max)

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands", fontsize=9)
    ax_all.set_ylabel(f"{flux_label} / median", fontsize=8)
    ax_all.legend(fontsize=7, ncol=3, loc="best")
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)
    # force a common x-scale
    ax_all.set_xlim(dt_min, dt_max)
    ax_all.set_ylim(yscale_min, yscale_max)

    return t0_date


def _add_date_axis(ax, dt_array, t0_mjd):
    """
    Add a secondary x-axis on top of *ax* showing calendar dates.

    Strategy: pick at most 5 evenly spaced tick positions from the data
    range, compute their MJD, convert to ISO date strings, and set them
    as a secondary x-axis twin.

    Parameters
    ----------
    ax       : matplotlib Axes
    dt_array : 1-D array of Δt [days] already plotted on ax
    t0_mjd   : float  — reference MJD (global first alert)
    """
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return

    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        n_ticks = 1
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)

    tick_mjd = t0_mjd + tick_dt
    tick_dates = mjd_to_datestr(tick_mjd)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


print("plot_relative_flux_object() and _add_date_axis() defined.")

### 6.1 Plot PSF relative flux

In [ ]:
# ── Main loop: PSF relative flux ───────────────────────────────────────────────
n_subplots = len(BANDS) + 1  # 6 bands + combined

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        sharey=False,
        constrained_layout=True,
    )

    t0_date = plot_relative_flux_object(
        obj_id,
        df_obj,
        flux_col=COL_PSF,
        flux_err_col=COL_PSFERR,
        flux_label="psfFlux",
        field_col="field",
        ax_rows=axes,
    )
    # field = df_obj["field"]
    field_vals = df_obj["field"].dropna().unique() if "field" in df_obj.columns else []
    field_str = field_vals[0] if len(field_vals) > 0 else "?"

    # Common x-label on bottom axes
    for ax in axes:
        ax.set_xlabel("Δt [days] from first alert", fontsize=8)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id} | {field_str} |  N={n_alerts} alerts  |  t₀={t0_date}",
        fontsize=11,
        y=1.02,
    )

    savefig(f"relflux_psf_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 7. Loop: AP-flux relative light curves

Same layout as Section 6, but using `apFlux` instead of `psfFlux`.

In [ ]:
if not has_ap:
    print("apFlux columns not available — skipping AP relative flux plots.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        n_alerts = len(df_obj)

        fig, axes = plt.subplots(
            1,
            n_subplots,
            figsize=(3.2 * n_subplots, 4.5),
            sharey=False,
            constrained_layout=True,
        )

        t0_date = plot_relative_flux_object(
            obj_id,
            df_obj,
            flux_col=COL_AP,
            flux_err_col=COL_APERR,
            flux_label="apFlux",
            field_col="field",
            ax_rows=axes,
        )

        for ax in axes:
            ax.set_xlabel("Δt [days] from first alert", fontsize=8)

        fig.suptitle(
            f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  N={n_alerts} alerts  |  t₀={t0_date}  [apFlux]",
            fontsize=11,
            y=1.05,
        )

        savefig(f"relflux_ap_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
        plt.show()
        # plt.close(fig)

## 8. Loop: PSF-ratio vs. AP-ratio comparison per band

For each DIA object and each band, we overlay:
- `psfFlux / median(psfFlux)` (filled circles)
- `apFlux  / median(apFlux)`  (open triangles)

Same 7-panel layout: `u | g | r | i | z | y | all-bands`.

In [ ]:
def plot_psf_vs_ap_object(obj_id, df_obj, axes):
    """
    Overlay PSF and AP relative fluxes for one DIA object.

    Parameters
    ----------
    obj_id : int/str   — diaObjectId
    df_obj : DataFrame — rows for this object
    axes   : list of 7 Axes

    Returns
    -------
    t0_date : str — ISO date of the first alert
    """
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)

        if len(df_b) == 0:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        dt = df_b[COL_MJD].values - t0_mjd
        color = BAND_COLORS.get(band, "k")

        # PSF ratio
        psf_ratio, psf_err, _, sigma_psf = compute_relative_flux(
            df_b[COL_PSF].values, df_b[COL_PSFERR].values
        )
        ax.errorbar(
            dt,
            psf_ratio,
            yerr=psf_err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=2,
            alpha=0.85,
            label=f"psf σ={sigma_psf:.3f}",
        )

        # AP ratio (if available)
        if has_ap:
            ap_ratio, ap_err, _, sigma_ap = compute_relative_flux(df_b[COL_AP].values, df_b[COL_APERR].values)
            ax.errorbar(
                dt,
                ap_ratio,
                yerr=ap_err,
                fmt="^",
                ms=4,
                color=color,
                ecolor=color,
                elinewidth=0.8,
                capsize=2,
                alpha=0.55,
                mfc="none",
                label=f"ap  σ={sigma_ap:.3f}",
            )

        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}", fontsize=9)
        ax.set_ylabel("flux / median(flux)", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        _add_date_axis(ax, dt, t0_mjd)

    # ── Combined panel ──────────────────────────────────────────────────────────
    ax_all = axes[-1]
    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        color = BAND_COLORS.get(band, "k")

        psf_ratio, psf_err, _, _ = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
        ax_all.errorbar(
            dt,
            psf_ratio,
            yerr=psf_err,
            fmt="o",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.75,
            label=f"{band} psf",
        )

        if has_ap:
            ap_ratio, ap_err, _, _ = compute_relative_flux(df_b[COL_AP].values, df_b[COL_APERR].values)
            ax_all.errorbar(
                dt,
                ap_ratio,
                yerr=ap_err,
                fmt="^",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.6,
                capsize=2,
                alpha=0.45,
                mfc="none",
                label=f"{band} ap",
            )

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — psf (●) vs ap (△)", fontsize=9)
    ax_all.set_ylabel("flux / median(flux)", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    return t0_date


print("plot_psf_vs_ap_object() defined.")

In [ ]:
# ── Main loop: PSF vs AP comparison ──────────────────────────────────────────
for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        sharey=False,
        constrained_layout=True,
    )

    t0_date = plot_psf_vs_ap_object(obj_id, df_obj, axes)

    for ax in axes:
        ax.set_xlabel("Δt [days] from first alert", fontsize=8)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  N={n_alerts} alerts  |  t₀={t0_date}\n"
        "psfFlux/median (●) vs apFlux/median (△)",
        fontsize=11,
        y=1.07,
    )

    savefig(f"psf_vs_ap_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 9. Summary table: per-object, per-band σ(ratio)

For each top-ranked object, we tabulate the RMS scatter of the relative flux ratio,
both for PSF and AP fluxes.  This serves as a quick variability / calibration diagnostic.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    n_total = len(df_obj)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": n_total,
        "t0_date": t0_date,
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"sigma_psf_{band}"] = np.nan
            row[f"sigma_ap_{band}"] = np.nan
            continue

        _, _, _, sigma_psf = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
        row[f"sigma_psf_{band}"] = round(sigma_psf, 4)

        if has_ap:
            _, _, _, sigma_ap = compute_relative_flux(df_b[COL_AP].values, df_b[COL_APERR].values)
            row[f"sigma_ap_{band}"] = round(sigma_ap, 4)
        else:
            row[f"sigma_ap_{band}"] = np.nan

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table of relative-flux scatter per object & band:")
display(df_summary)

In [ ]:
# Save summary table
summary_path = os.path.join(DIR_FIGS, f"sigma_ratio_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")